In [ ]:
# =============================================================
# NOTEBOOK 03 — EXPERIMENTATION
# En MLOps, la experimentación es el proceso de probar múltiples modelos
# con distintas configuraciones (hiperparámetros) para encontrar el mejor.
# Este notebook usa MLflow para TRACKEAR cada experimento:
# qué parámetros se usaron, qué métricas se obtuvieron, y qué modelo se guardó.
# Sin tracking, no hay reproducibilidad — y sin reproducibilidad, no hay MLOps.
# =============================================================

import pandas as pd                                    # Para cargar y manipular el dataset
import numpy as np                                     # Para operaciones numéricas (ej: calcular RMSE con sqrt)
import joblib                                          # Para serializar (guardar) modelos entrenados en disco
import mlflow                                          # Framework de tracking de experimentos ML
import mlflow.sklearn                                  # Módulo específico de MLflow para modelos de scikit-learn
from sklearn.model_selection import train_test_split, GridSearchCV  # train_test_split: divide datos en train/test
                                                                      # GridSearchCV: búsqueda exhaustiva de hiperparámetros
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor  # Modelos de ensemble (combinan múltiples árboles)
from sklearn.linear_model import LinearRegression      # Modelo de regresión lineal (baseline simple)
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score  # Métricas para evaluar modelos de regresión
import xgboost as xgb                                  # XGBoost: algoritmo de gradient boosting muy popular en competencias ML
import yaml                                            # Para leer/escribir archivos YAML (configuraciones)
import os                                              # Para operaciones del sistema de archivos (crear directorios, etc.)
import logging                                         # Para registrar mensajes de log durante el entrenamiento
from contextlib import nullcontext                     # Para usar 'with' de forma condicional (si MLflow está activo o no)
import matplotlib.pyplot as plt                        # Para visualizar comparaciones entre modelos

logging.basicConfig(level=logging.INFO)  # Configura el sistema de logging para mostrar mensajes de nivel INFO y superiores
                                          # Niveles: DEBUG < INFO < WARNING < ERROR < CRITICAL
logger = logging.getLogger()             # Obtiene el logger raíz para usarlo en el notebook

mlflow_tracking_uri = 'http://localhost:5555'  # URI del servidor MLflow donde se guardarán los experimentos
                                                # Si MLflow no está corriendo en este puerto, el tracking fallará
                                                # En producción, esto apuntaría a un servidor MLflow centralizado


In [ ]:
# Load dataset
# En MLOps, cargamos el dataset con features ya engineered (del notebook 02).
# El pipeline es: raw → cleaned → featured → train/test split → modelos
data_path = '../data/processed/featured_house_data.csv'  # Ruta al dataset con features engineered
data = pd.read_csv(data_path)                             # Carga el CSV como DataFrame

X = data.drop('price', axis=1)  # X = features (todas las columnas EXCEPTO el target)
                                  # .drop('price', axis=1): elimina la columna 'price'
                                  # axis=1 significa que eliminamos una columna (axis=0 sería una fila)

y = data['price']                 # y = target variable (lo que queremos predecir: el precio)
                                  # En ML supervisado, X son las entradas y 'y' es la salida esperada

X_train, X_test, y_train, y_test = train_test_split(
    X, y,                         # Los datos a dividir
    test_size=0.2,                # 20% de los datos van al conjunto de test, 80% al de entrenamiento
    random_state=42               # Semilla aleatoria fija → garantiza reproducibilidad
                                  # Con random_state=42, siempre obtenemos la misma división
)                                 # Resultado: 4 variables con los datos divididos


In [ ]:
from sklearn.feature_selection import RFE  # RFE: Recursive Feature Elimination
                                            # Algoritmo que elimina features de forma iterativa
                                            # manteniendo solo las más importantes para el modelo
from xgboost import XGBRegressor           # Importación directa de XGBRegressor desde el módulo xgboost

# Use XGBoost for RFE to stay consistent
# En MLOps, usamos el mismo algoritmo para RFE que para el modelo final
# para que la selección de features sea coherente con el modelo que se desplegará
xgb_model = XGBRegressor(objective='reg:squarederror')  # Crea un modelo XGBoost para regresión
                                                          # objective='reg:squarederror': minimiza el error cuadrático medio
                                                          # Es la función de pérdida estándar para regresión
xgb_model.fit(X_train, y_train)  # Entrena el modelo XGBoost con los datos de entrenamiento
                                  # .fit() ajusta los parámetros del modelo a los datos

# RFE
rfe_selector = RFE(
    estimator=xgb_model,          # Usa el modelo XGBoost entrenado para evaluar la importancia de features
    n_features_to_select=10       # Selecciona las 10 features más importantes (descarta el resto)
)
rfe_selector.fit(X_train, y_train)  # Ejecuta el algoritmo RFE: entrena, evalúa importancia, elimina la peor feature, repite

rfe_selected_features = X.columns[rfe_selector.support_]   # .support_ es un array booleano: True para features seleccionadas
                                                             # X.columns[...] filtra los nombres de columnas donde support_=True
rfe_ignored_features = X.columns[~rfe_selector.support_]   # '~' invierte el booleano: True donde support_=False
                                                             # = features que RFE descartó por ser poco informativas

print("✅ Top 10 Selected Features by RFE:")  # Encabezado
for feature in rfe_selected_features:          # Itera sobre las features seleccionadas
    print(f" - {feature}")                     # Imprime cada feature seleccionada

print("\n❌ Features Ignored by RFE:")         # Encabezado para features descartadas
for feature in rfe_ignored_features:           # Itera sobre las features descartadas
    print(f" - {feature}")                     # Imprime cada feature descartada

# Store for config
# En MLOps, guardamos qué features se seleccionaron para que el pipeline de producción
# use exactamente las mismas features al hacer predicciones en tiempo real
selected_features_dict = {
    'rfe': list(rfe_selected_features)  # Convierte el índice a lista Python para poder serializarlo en YAML/JSON
}

# Filter datasets to use only selected features for experimentation
X_train = X_train[rfe_selected_features]  # Reemplaza X_train con solo las 10 features seleccionadas
X_test = X_test[rfe_selected_features]    # Hace lo mismo con X_test — CRÍTICO: train y test deben tener las mismas columnas


In [ ]:
# Optional MLflow setup
# En MLOps, MLflow es el sistema de tracking de experimentos.
# Registra: qué parámetros usaste, qué métricas obtuviste, y guarda el modelo entrenado.
# Esto permite comparar experimentos y reproducir cualquier resultado pasado.

if mlflow_tracking_uri:                                          # Solo configura MLflow si se definió una URI
    mlflow.set_tracking_uri(mlflow_tracking_uri)                 # Apunta MLflow al servidor en localhost:5555
                                                                  # Todos los runs se guardarán en ese servidor
    mlflow.set_experiment("House Price Prediction Experiment")   # Crea (o selecciona) un experimento con ese nombre
                                                                  # Un experimento agrupa múltiples 'runs' (entrenamientos)


In [ ]:
# Define models and hyperparameter grids
# En MLOps, comparar múltiples algoritmos es una práctica estándar.
# No asumimos que un algoritmo es mejor — lo medimos con datos.

models = {
    'LinearRegression': LinearRegression(),                          # Modelo lineal simple — sirve como BASELINE
                                                                      # Si un modelo complejo no supera el baseline, algo está mal
    'RandomForest': RandomForestRegressor(),                         # Ensemble de árboles de decisión — robusto y versátil
    'GradientBoosting': GradientBoostingRegressor(),                 # Árboles entrenados secuencialmente, cada uno corrige al anterior
    'XGBoost': xgb.XGBRegressor(objective='reg:squarederror')       # Implementación optimizada de Gradient Boosting — muy popular
}

model_grids = {                         # Diccionario de hiperparámetros a probar para cada modelo
                                         # GridSearchCV probará TODAS las combinaciones posibles
    'LinearRegression': {},             # LinearRegression no tiene hiperparámetros relevantes → grid vacío
    'RandomForest': {
        'n_estimators': [100, 150],     # Número de árboles en el bosque: probar 100 y 150
        'max_depth': [None, 10, 20]     # Profundidad máxima de cada árbol: None=sin límite, 10, o 20
                                         # Total combinaciones: 2 × 3 = 6 modelos a entrenar
    },
    'GradientBoosting': {
        'n_estimators': [100, 250],     # Número de árboles (boosting rounds)
        'learning_rate': [0.1, 0.05],  # Tasa de aprendizaje: qué tanto corrige cada árbol al anterior
                                         # Valores más bajos = más preciso pero más lento
        'max_depth': [3, 10]            # Profundidad de cada árbol individual
                                         # Total combinaciones: 2 × 2 × 2 = 8 modelos a entrenar
    },
    'XGBoost': {
        'n_estimators': [100, 150],     # Número de rondas de boosting
        'learning_rate': [0.1, 0.05],  # Tasa de aprendizaje (también llamado 'eta' en XGBoost)
        'max_depth': [3, 10]            # Profundidad máxima de cada árbol
                                         # Total combinaciones: 2 × 2 × 2 = 8 modelos a entrenar
    }
}


In [ ]:
def evaluate_model_with_gridsearch(name, model, grid, X_train, y_train, X_test, y_test):
    # Función que entrena un modelo con GridSearchCV y devuelve sus métricas
    # En MLOps, encapsular la lógica de evaluación en funciones garantiza consistencia
    # entre todos los modelos que comparamos

    if grid:  # Si el grid de hiperparámetros no está vacío...
        clf = GridSearchCV(
            model,              # El modelo base a optimizar
            grid,               # El diccionario de hiperparámetros a probar
            cv=3,               # Cross-validation con 3 folds: divide train en 3 partes,
                                # entrena en 2 y valida en 1, repite 3 veces
            scoring='r2',       # Métrica para seleccionar el mejor modelo: R² (coeficiente de determinación)
                                # R²=1 es perfecto, R²=0 es tan malo como predecir siempre el promedio
            n_jobs=-1           # Usa todos los núcleos del CPU disponibles para paralelizar la búsqueda
        )
        clf.fit(X_train, y_train)      # Ejecuta la búsqueda: prueba todas las combinaciones de hiperparámetros
        best_model = clf.best_estimator_  # .best_estimator_: el modelo con los mejores hiperparámetros encontrados
        best_params = clf.best_params_    # .best_params_: diccionario con los mejores hiperparámetros
    else:  # Si el grid está vacío (ej: LinearRegression)
        model.fit(X_train, y_train)    # Entrena el modelo directamente sin búsqueda de hiperparámetros
        best_model = model             # El modelo entrenado es el 'mejor' (único)
        best_params = model.get_params()  # .get_params(): devuelve los parámetros por defecto del modelo

    y_pred = best_model.predict(X_test)  # Genera predicciones sobre el conjunto de TEST (datos que el modelo nunca vio)
                                          # CRÍTICO: siempre evaluar en test, nunca en train (eso sería trampa)

    mae = mean_absolute_error(y_test, y_pred)   # MAE: error absoluto medio — promedio de |real - predicho|
                                                  # Fácil de interpretar: "en promedio, me equivoco en $X"
    mse = mean_squared_error(y_test, y_pred)     # MSE: error cuadrático medio — penaliza más los errores grandes
    rmse = np.sqrt(mse)                           # RMSE: raíz del MSE — en las mismas unidades que el target (precio)
    r2 = r2_score(y_test, y_pred)                # R²: qué proporción de la varianza del target explica el modelo
                                                  # R²=0.85 significa que el modelo explica el 85% de la variación en precios

    return {          # Devuelve un diccionario con todas las métricas y el modelo entrenado
        'mae': mae,
        'mse': mse,
        'rmse': rmse,
        'r2': r2,
        'model': best_model,   # El modelo entrenado — lo necesitamos para guardarlo después
        'params': best_params  # Los mejores hiperparámetros — los logueamos en MLflow
    }

print("MLflow tracking URI:", mlflow_tracking_uri)  # Confirma la URI de MLflow configurada


In [ ]:
results = {}  # Diccionario vacío donde guardaremos los resultados de cada modelo

# En MLOps, el tracking con MLflow es fundamental para:
# 1. Comparar experimentos entre sí
# 2. Reproducir cualquier experimento pasado
# 3. Registrar el modelo ganador en el Model Registry

with mlflow.start_run(run_name="model_comparison") if mlflow_tracking_uri else nullcontext():
    # mlflow.start_run(): inicia un 'run' padre que agrupa todos los modelos
    # run_name="model_comparison": nombre descriptivo para identificar este experimento en la UI de MLflow
    # 'if mlflow_tracking_uri else nullcontext()': si no hay URI de MLflow, usa un contexto vacío (no hace nada)
    # Esto hace el código flexible: funciona con o sin MLflow

    for name, model in models.items():  # Itera sobre cada modelo del diccionario 'models'
                                         # name = 'LinearRegression', 'RandomForest', etc.
                                         # model = la instancia del modelo

        logger.info(f"Training {name}...")  # Log informativo: indica qué modelo se está entrenando ahora

        with mlflow.start_run(run_name=name, nested=True) if mlflow_tracking_uri else nullcontext():
            # nested=True: crea un 'run' hijo dentro del run padre 'model_comparison'
            # Esto organiza los experimentos en una jerarquía en la UI de MLflow

            evaluation = evaluate_model_with_gridsearch(
                name, model, model_grids[name],  # Pasa el nombre, modelo y su grid de hiperparámetros
                X_train, y_train, X_test, y_test  # Datos de entrenamiento y test
            )
            results[name] = evaluation  # Guarda los resultados de este modelo en el diccionario

            if mlflow_tracking_uri:  # Solo loguea en MLflow si está configurado
                mlflow.log_params(evaluation['params'])  # Registra los hiperparámetros usados
                                                          # Aparecen en la columna 'Parameters' de la UI de MLflow
                mlflow.log_metrics({                     # Registra las métricas de evaluación
                    'mae': evaluation['mae'],             # Error absoluto medio
                    'mse': evaluation['mse'],             # Error cuadrático medio
                    'rmse': evaluation['rmse'],           # Raíz del error cuadrático medio
                    'r2': evaluation['r2']                # Coeficiente de determinación
                })
                mlflow.sklearn.log_model(                # Guarda el modelo entrenado como artefacto en MLflow
                    evaluation['model'],                  # El objeto del modelo entrenado
                    artifact_path=name.lower().replace(" ", "_")  # Ruta donde se guarda en MLflow
                                                                    # ej: 'randomforest', 'xgboost'
                )

            print(f"{name} R2: {evaluation['r2']:.4f}, RMSE: {evaluation['rmse']:.2f}")
            # Imprime un resumen rápido: nombre del modelo, R² con 4 decimales, RMSE con 2 decimales


In [ ]:
def plot_model_metrics(results, metric='r2'):
    # Función para visualizar y comparar una métrica entre todos los modelos
    # En MLOps, la visualización de métricas facilita la toma de decisiones
    # sobre qué modelo desplegar a producción

    names = list(results.keys())                          # Lista de nombres de modelos: ['LinearRegression', 'RandomForest', ...]
    values = [results[name][metric] for name in names]    # List comprehension: extrae el valor de la métrica para cada modelo
                                                           # ej: [0.72, 0.89, 0.91, 0.93] para R²

    plt.figure(figsize=(10, 6))         # Figura de 10x6 pulgadas
    plt.bar(names, values)              # Gráfico de barras: eje X = nombres de modelos, eje Y = valor de la métrica
    plt.title(f'Model Comparison by {metric.upper()}')  # Título dinámico: 'Model Comparison by R2', 'by RMSE', etc.
    plt.xlabel('Model')                 # Etiqueta eje X
    plt.ylabel(metric.upper())          # Etiqueta eje Y con la métrica en mayúsculas
    plt.grid(True)                      # Muestra una grilla de fondo para facilitar la lectura de valores
    plt.show()                          # Renderiza y muestra el gráfico

# Example usage:
# En MLOps, comparamos múltiples métricas porque cada una cuenta una historia diferente:
plot_model_metrics(results, metric='r2')    # R²: qué tan bien explica el modelo la varianza del precio
plot_model_metrics(results, metric='rmse')  # RMSE: error promedio en las mismas unidades que el precio
plot_model_metrics(results, metric='mae')   # MAE: error absoluto medio — más robusto a outliers que RMSE


In [ ]:
import yaml  # Para serializar la configuración del modelo en formato YAML
              # YAML es legible por humanos — ideal para archivos de configuración en MLOps

# Save model config with selected features
# En MLOps, guardar la configuración del mejor modelo es CRÍTICO para reproducibilidad.
# El script de entrenamiento en producción (train.py) leerá este archivo para saber
# qué algoritmo usar, con qué hiperparámetros, y con qué features.

# Display information about the best model
best_model_name = max(results, key=lambda x: results[x]['r2'])  # Encuentra el modelo con el R² más alto
                                                                   # max() con key=lambda: compara por el valor de R²
                                                                   # lambda x: results[x]['r2'] = función anónima que extrae el R²

best_model = results[best_model_name]['model']    # Obtiene el objeto del modelo ganador
best_params = best_model.get_params()             # .get_params(): devuelve todos los hiperparámetros del modelo ganador
best_r2 = float(results[best_model_name]['r2'])   # float(): convierte a float nativo de Python (necesario para serializar en YAML)
best_mae = float(results[best_model_name]['mae'])  # Lo mismo para MAE
best_rmse = float(results[best_model_name]['rmse'])  # Lo mismo para RMSE

print(f"🏆 Best Model: {best_model_name}")   # Imprime el nombre del modelo ganador
print(f"   R² Score: {best_r2:.4f}")         # R² con 4 decimales
print(f"   MAE: {best_mae:.2f}")             # MAE con 2 decimales
print(f"   RMSE: {best_rmse:.2f}")           # RMSE con 2 decimales

model_config = {                              # Diccionario con toda la información del mejor modelo
    'model': {
        'name': 'house_price_model',          # Nombre del modelo para identificarlo en producción
        'best_model': best_model_name,        # Algoritmo ganador (ej: 'XGBoost')
        'parameters': best_params,            # Hiperparámetros óptimos encontrados por GridSearchCV
        'r2_score': best_r2,                  # Métrica de rendimiento para referencia
        'mae': best_mae,                      # MAE para referencia
        'target_variable': 'price',           # Variable que predice el modelo
        'feature_sets': selected_features_dict  # Features seleccionadas por RFE — el pipeline de producción
                                                 # debe usar EXACTAMENTE estas features al hacer predicciones
    }
}

config_path = '../configs/model_config.yaml'              # Ruta donde se guardará la configuración
os.makedirs(os.path.dirname(config_path), exist_ok=True)  # Crea el directorio 'configs/' si no existe
                                                           # exist_ok=True: no falla si el directorio ya existe

with open(config_path, 'w') as f:   # Abre el archivo en modo escritura ('w')
    yaml.dump(model_config, f)       # yaml.dump(): serializa el diccionario Python como YAML y lo escribe en el archivo
                                     # El resultado es un archivo legible que documenta el modelo ganador

print(f"Saved model config to {config_path}")  # Confirma que la configuración se guardó correctamente
# En MLOps, este archivo YAML es el 'contrato' entre la experimentación y el pipeline de producción
